### Gathering Values for Analysis
We process each reusting file here to show the grouped datasets generated by our approach and the corresponding values for the defined metrics in the paper. We document each scenario's alerts with following values:
- N: Total number of alerts in the scenario
- P: Number of clusters in each scenario
- R: Number of unclustered alerts in each scenario
- G: Total number of inspection units after grouping in each scenario. G = P + R

We further calculate the metrics of Coverage and Workload reduction


In [1]:
import os
import glob
import pandas as pd


OUTPUT_DIR = "/home/arch/Downloads/Alert-grouping-pipeline/output"


def add_is_cluster(df):
    """Add a binary indicator for whether a group is a cluster."""
    df = df.copy()
    df["is_cluster"] = (df["group_size"] > 1).astype(int)
    return df


def calculate_scenario_metrics(scenario_name, output_dir=OUTPUT_DIR):
    """
    Calculate workload-reduction and coverage metrics for one scenario.

    Parameters
    ----------
    scenario_name : str
        Scenario name, e.g. 'fox_alerts'.
    output_dir : str
        Directory containing the grouped and validity files.

    Returns
    -------
    dict
        N, P, R, G, workload reduction, and coverage.
    """

    grouped_file = os.path.join(
        output_dir,
        f"{scenario_name}_grouped.csv"
    )

    validity_file = os.path.join(
        output_dir,
        f"{scenario_name}_group_validity.csv"
    )

    # Read files
    grouped_df = pd.read_csv(grouped_file)
    validity_results = pd.read_csv(validity_file)

    # Add cluster indicator
    validity_results = add_is_cluster(validity_results)

    # P = number of clusters
    P = (validity_results["is_cluster"] == 1).sum()

    # R = number of unclustered alerts
    R = (validity_results["is_cluster"] == 0).sum()

    # N = total number of alerts
    N = len(grouped_df)

    # G = number of inspection units after clustering
    G = P + R

    # Workload reduction
    workload_reduction = 1 - (G / N)

    # Coverage
    coverage = (N - R) / N

    return {
        "scenario": scenario_name,
        "N": N,
        "P": P,
        "R": R,
        "G": G,
        "workload_reduction": workload_reduction,
        "coverage": coverage
    }


def calculate_all_scenarios(output_dir=OUTPUT_DIR):
    """
    Calculate workload-reduction and coverage metrics
    for all scenarios found in the output directory.
    """

    # Find all validity files
    validity_files = glob.glob(
        os.path.join(output_dir, "*_group_validity.csv")
    )

    results = []

    for validity_file in validity_files:

        filename = os.path.basename(validity_file)

        # Extract scenario name
        scenario_name = filename.replace(
            "_group_validity.csv", ""
        )

        try:
            metrics = calculate_scenario_metrics(
                scenario_name,
                output_dir
            )

            results.append(metrics)

        except Exception as e:
            print(
                f"Could not process {scenario_name}: {e}"
            )

    return pd.DataFrame(results)




In [4]:
# Calculate results for all 8 scenarios
results = calculate_all_scenarios()

# Add percentage versions
results["workload_reduction_percent"] = (
    results["workload_reduction"] * 100
)

results["coverage_percent"] = (
    results["coverage"] * 100
)

# Display results
print(
    results[
        [
            "scenario",
            "N",
            "P",
            "R",
            "G",
            "workload_reduction",
            "coverage",
            "workload_reduction_percent",
            "coverage_percent"
        ]
    ].to_string(index=False)
)


/tmp/ipykernel_24237/150965653.py:44: DtypeWarning: Columns (12,14) have mixed types. Specify dtype option on import or set low_memory=False.
  grouped_df = pd.read_csv(grouped_file)


              scenario      N     P   R     G  workload_reduction  coverage  workload_reduction_percent  coverage_percent
        wheeler_alerts 616161 19341 477 19818            0.967836  0.999226                   96.783633         99.922585
       wardbeck_alerts  91257 11314 498 11812            0.870563  0.994543                   87.056335         99.454288
           shaw_alerts  70782  9421 540  9961            0.859272  0.992371                   85.927213         99.237094
       harrison_alerts 593948 32269 223 32492            0.945295  0.999625                   94.529487         99.962455
         wilson_alerts 634246 24092 491 24583            0.961241  0.999226                   96.124059         99.922585
         santos_alerts 130779 21811 180 21991            0.831846  0.998624                   83.184609         99.862363
russellmitchell_alerts  45544  5029 256  5285            0.883958  0.994379                   88.395837         99.437906
            fox_alerts 4

/tmp/ipykernel_24237/150965653.py:44: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  grouped_df = pd.read_csv(grouped_file)
